# Step 2: Vehicle and Road User Detection
### Traffic Violation Detection System — Phase 1

**Objective:** Detect and localize vehicles, riders, drivers, and pedestrians using YOLOv8 pretrained models (Ultralytics).

---
**Models Available (from your local collection):**
- `yolov8n.pt` / `yolov8s.pt` / `yolov8m.pt` / `yolov8l.pt` — General object detection (COCO)
- `yolov8n-person.pt` — Specialized person detection
- `yolov8n-face.pt`, `yolov8m-face.pt` — Face detection (useful for driver/rider identification)

**Detection Targets:**
- 🚗 Vehicles: car, truck, bus, motorcycle, bicycle, auto-rickshaw
- 🧍 Road Users: pedestrian, rider, driver
- 🏍️ Riders: motorcycle/bicycle rider classification

## Cell 1 — Install Dependencies

In [ ]:
# Install required packages
!pip install ultralytics opencv-python-headless Pillow matplotlib seaborn pandas tqdm easyocr --quiet

print("✅ All packages installed successfully.")

## Cell 2 — Import Libraries

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.colors import to_rgba
import seaborn as sns
from PIL import Image
from pathlib import Path
from tqdm import tqdm
from collections import defaultdict
import warnings
import json
import time

from ultralytics import YOLO
import torch

warnings.filterwarnings('ignore')

# Check GPU availability
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"🖥️  Device: {device.upper()}")
if device == 'cuda':
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("   Running on CPU (inference will be slower)")

print(f"\n📦 Ultralytics version: {__import__('ultralytics').__version__}")
print(f"🔥 PyTorch version: {torch.__version__}")

## Cell 3 — Configuration & Class Mappings

In [ ]:
# ─────────────────────────────────────────────
# CONFIGURATION  — Edit these paths as needed
# ─────────────────────────────────────────────

# Path to your pretrained models folder
# Change this to the folder where your .pt files are stored
MODELS_DIR = "./models"          # e.g. "/content/models" for Colab

# Input: folder of traffic images OR a single image/video
INPUT_PATH = "./input"           # folder with traffic images

# Output folder for annotated results
OUTPUT_DIR = "./output/step2_detection"
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/annotated", exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/reports", exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/crops", exist_ok=True)

# Model to use for main detection
# Options from your collection: yolov8n, yolov8s, yolov8m, yolov8l
# 'n' = fastest, 'l' = most accurate
MAIN_MODEL    = os.path.join(MODELS_DIR, "yolov8m.pt")   # balanced accuracy/speed
PERSON_MODEL  = os.path.join(MODELS_DIR, "yolov8n-person.pt")

# Detection thresholds
CONF_THRESHOLD = 0.40   # minimum confidence score
IOU_THRESHOLD  = 0.45   # NMS IoU threshold
IMG_SIZE       = 640    # inference image size (640 or 1280 for better detail)

# ─────────────────────────────────────────────
# COCO class IDs relevant to traffic detection
# ─────────────────────────────────────────────
TRAFFIC_CLASS_IDS = {
    0:  "person",
    1:  "bicycle",
    2:  "car",
    3:  "motorcycle",
    5:  "bus",
    7:  "truck",
}

# Extended vehicle taxonomy for output reporting
VEHICLE_CLASSES   = {2: "car", 5: "bus", 7: "truck", 3: "motorcycle", 1: "bicycle"}
ROAD_USER_CLASSES = {0: "person"}

# Colour palette per class (BGR for OpenCV)
CLASS_COLORS_BGR = {
    "person":     (0,   200, 255),   # orange
    "bicycle":    (0,   255, 160),   # spring green
    "car":        (255, 100,   0),   # blue
    "motorcycle": (0,   100, 255),   # red-orange
    "bus":        (200,   0, 200),   # magenta
    "truck":      (50,  200,  50),   # green
}

print("✅ Configuration loaded.")
print(f"   Main model   : {MAIN_MODEL}")
print(f"   Person model : {PERSON_MODEL}")
print(f"   Output folder: {OUTPUT_DIR}")
print(f"   Confidence   : {CONF_THRESHOLD}")
print(f"   Tracked classes: {list(TRAFFIC_CLASS_IDS.values())}")

## Cell 4 — Load Pretrained Models

In [ ]:
def load_model(model_path, device):
    """Load a YOLO model, falling back to auto-download if file not found."""
    if os.path.exists(model_path):
        print(f"   Loading from disk : {model_path}")
        model = YOLO(model_path)
    else:
        # Fallback: download standard YOLOv8 from Ultralytics
        basename = os.path.basename(model_path)
        print(f"   ⚠️  File not found. Downloading: {basename}")
        model = YOLO(basename)
    model.to(device)
    return model

print("🔄 Loading models...")

# Primary detection model (vehicles + persons on COCO)
main_model = load_model(MAIN_MODEL, device)
print(f"   ✅ Main model loaded  → {main_model.model.__class__.__name__}")

# Specialised person / rider model for better recall on pedestrians
person_model = load_model(PERSON_MODEL, device)
print(f"   ✅ Person model loaded")

print("\n✅ All models ready.")

## Cell 5 — Image Preprocessing Utilities

In [ ]:
def preprocess_image(img_bgr, target_size=None):
    """
    Light preprocessing to improve detection quality:
    - CLAHE for low-light / uneven exposure
    - Optional resize (preserving aspect ratio)
    Returns the preprocessed BGR image.
    """
    # Convert to LAB and enhance luminance channel with CLAHE
    lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    l_eq = clahe.apply(l)
    enhanced = cv2.merge([l_eq, a, b])
    enhanced_bgr = cv2.cvtColor(enhanced, cv2.COLOR_LAB2BGR)

    # Optional resize (letterbox handled internally by YOLO)
    if target_size:
        h, w = enhanced_bgr.shape[:2]
        scale = target_size / max(h, w)
        nh, nw = int(h * scale), int(w * scale)
        enhanced_bgr = cv2.resize(enhanced_bgr, (nw, nh), interpolation=cv2.INTER_LINEAR)

    return enhanced_bgr


def load_image(path):
    """Load image via OpenCV; return (bgr_array, original_shape)."""
    img = cv2.imread(str(path))
    if img is None:
        raise FileNotFoundError(f"Cannot read image: {path}")
    return img, img.shape[:2]   # (H, W)


# Quick smoke test
dummy = np.zeros((480, 640, 3), dtype=np.uint8)
out   = preprocess_image(dummy)
print(f"✅ Preprocessing OK  →  input {dummy.shape}  →  output {out.shape}")

## Cell 6 — Core Detection Function

In [ ]:
def detect_traffic_objects(img_bgr, main_model, person_model,
                            conf=CONF_THRESHOLD, iou=IOU_THRESHOLD,
                            imgsz=IMG_SIZE):
    """
    Run dual-model detection:
      1. Main model  → vehicles + persons (COCO)
      2. Person model → additional pedestrian / rider recall
    Returns list of dicts: {class_id, class_name, category, conf, bbox (x1,y1,x2,y2)}
    """
    detections = []
    seen_boxes  = []   # for deduplication

    def iou_overlap(b1, b2):
        """Compute IoU between two (x1,y1,x2,y2) boxes."""
        ix1 = max(b1[0], b2[0]); iy1 = max(b1[1], b2[1])
        ix2 = min(b1[2], b2[2]); iy2 = min(b1[3], b2[3])
        inter = max(0, ix2-ix1) * max(0, iy2-iy1)
        a1 = (b1[2]-b1[0]) * (b1[3]-b1[1])
        a2 = (b2[2]-b2[0]) * (b2[3]-b2[1])
        return inter / (a1 + a2 - inter + 1e-6)

    def add_detections(results, allowed_ids):
        for r in results:
            boxes = r.boxes
            if boxes is None:
                continue
            for box in boxes:
                cls_id = int(box.cls[0])
                if cls_id not in allowed_ids:
                    continue
                conf_val = float(box.conf[0])
                x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
                bbox = (x1, y1, x2, y2)

                # Deduplicate with seen boxes (IoU > 0.5)
                dup = any(iou_overlap(bbox, s) > 0.50 for s in seen_boxes)
                if dup:
                    continue

                seen_boxes.append(bbox)
                cls_name = TRAFFIC_CLASS_IDS.get(cls_id, f"cls_{cls_id}")
                category = "vehicle" if cls_id in VEHICLE_CLASSES else "road_user"

                # Refine road_user → rider if on motorcycle/bicycle
                detections.append({
                    "class_id":   cls_id,
                    "class_name": cls_name,
                    "category":   category,
                    "confidence": round(conf_val, 4),
                    "bbox":       bbox,
                })

    # --- Run main model (vehicles + persons) ---
    res_main = main_model.predict(
        source=img_bgr, conf=conf, iou=iou, imgsz=imgsz,
        classes=list(TRAFFIC_CLASS_IDS.keys()), verbose=False
    )
    add_detections(res_main, TRAFFIC_CLASS_IDS)

    # --- Run person model for additional pedestrian recall ---
    res_person = person_model.predict(
        source=img_bgr, conf=conf, iou=iou, imgsz=imgsz,
        verbose=False
    )
    # Person-model outputs class 0 = person
    add_detections(res_person, {0: "person"})

    return detections


print("✅ Detection function defined.")

## Cell 7 — Rider / Vehicle Association (Spatial Proximity)

In [ ]:
def associate_riders(detections):
    """
    Post-processing: if a 'person' bbox overlaps significantly with a
    motorcycle or bicycle bbox, relabel the person as 'rider'.
    This is critical for triple-riding detection later.
    """
    two_wheelers = [d for d in detections if d["class_name"] in ("motorcycle", "bicycle")]
    persons      = [d for d in detections if d["class_name"] == "person"]

    def overlap_ratio(pb, vb):
        """What fraction of person bbox overlaps with vehicle bbox."""
        ix1 = max(pb[0], vb[0]); iy1 = max(pb[1], vb[1])
        ix2 = min(pb[2], vb[2]); iy2 = min(pb[3], vb[3])
        inter = max(0, ix2-ix1) * max(0, iy2-iy1)
        p_area = max(1, (pb[2]-pb[0]) * (pb[3]-pb[1]))
        return inter / p_area

    for person in persons:
        for vehicle in two_wheelers:
            ratio = overlap_ratio(person["bbox"], vehicle["bbox"])
            if ratio > 0.35:          # person is 35 %+ inside the vehicle box
                person["class_name"] = "rider"
                person["category"]   = "road_user"
                person["vehicle_id"] = id(vehicle)
                break

    # Count riders per vehicle for triple-riding flag
    vehicle_rider_count = defaultdict(int)
    for person in persons:
        if person["class_name"] == "rider":
            vid = person.get("vehicle_id")
            if vid:
                vehicle_rider_count[vid] += 1

    # Annotate vehicles with their rider count
    for vehicle in two_wheelers:
        rc = vehicle_rider_count.get(id(vehicle), 0)
        vehicle["rider_count"] = rc
        if rc >= 3:
            vehicle["triple_riding_flag"] = True

    return detections


print("✅ Rider association function defined.")

## Cell 8 — Annotation / Drawing Utilities

In [ ]:
def draw_detections(img_bgr, detections, font_scale=0.55, thickness=2):
    """
    Draw bounding boxes + labels on the image.
    Returns annotated copy (does NOT modify original).
    """
    annotated = img_bgr.copy()
    h, w = annotated.shape[:2]

    for det in detections:
        x1, y1, x2, y2 = det["bbox"]
        cls_name = det["class_name"]
        conf_val = det["confidence"]

        color = CLASS_COLORS_BGR.get(cls_name, (180, 180, 180))

        # Box
        cv2.rectangle(annotated, (x1, y1), (x2, y2), color, thickness)

        # Label text
        label = f"{cls_name} {conf_val:.2f}"
        if det.get("triple_riding_flag"):
            label += " ⚠️TRIPLE"

        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, font_scale, 1)
        ty = max(y1 - 6, th + 4)

        # Background pill for label
        cv2.rectangle(annotated, (x1, ty - th - 4), (x1 + tw + 6, ty + 2), color, -1)
        cv2.putText(annotated, label, (x1 + 3, ty - 2),
                    cv2.FONT_HERSHEY_SIMPLEX, font_scale, (255, 255, 255), 1, cv2.LINE_AA)

    # Overlay summary counts top-left
    counts = defaultdict(int)
    for d in detections:
        counts[d["class_name"]] += 1

    summary_lines = [f"Total detected: {len(detections)}"] + \
                    [f"  {k}: {v}" for k, v in sorted(counts.items())]

    y_off = 24
    for line in summary_lines:
        cv2.putText(annotated, line, (10, y_off),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2, cv2.LINE_AA)
        y_off += 22

    return annotated


def show_image(img_bgr, title="Detection Result", figsize=(14, 8)):
    """Display a BGR image inline in the notebook."""
    rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    fig, ax = plt.subplots(figsize=figsize)
    ax.imshow(rgb)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.axis('off')
    plt.tight_layout()
    plt.show()


print("✅ Drawing utilities defined.")

## Cell 9 — Single Image Detection Pipeline

In [ ]:
def process_single_image(image_path, main_model, person_model,
                          save_annotated=True, show=True):
    """
    Full pipeline for one image:
      load → preprocess → detect → associate riders → annotate → save
    Returns (annotated_bgr, detections_list, metadata_dict)
    """
    image_path = Path(image_path)
    t0 = time.time()

    # 1. Load
    img_bgr, (h, w) = load_image(image_path)

    # 2. Preprocess (CLAHE enhancement)
    img_pre = preprocess_image(img_bgr)

    # 3. Detect
    detections = detect_traffic_objects(img_pre, main_model, person_model)

    # 4. Associate riders with vehicles
    detections = associate_riders(detections)

    # 5. Annotate
    annotated = draw_detections(img_bgr, detections)  # draw on original colours

    elapsed = time.time() - t0

    # 6. Save
    out_path = None
    if save_annotated:
        out_path = os.path.join(OUTPUT_DIR, "annotated", image_path.name)
        cv2.imwrite(out_path, annotated)

    # 7. Metadata
    counts = defaultdict(int)
    for d in detections:
        counts[d["class_name"]] += 1

    meta = {
        "filename":    image_path.name,
        "image_size":  f"{w}x{h}",
        "inference_ms": round(elapsed * 1000, 1),
        "total_detected": len(detections),
        **{f"count_{k}": v for k, v in counts.items()},
    }

    # 8. Display inline
    if show:
        title = (f"{image_path.name}  |  "
                 f"{len(detections)} detections  |  {elapsed*1000:.0f} ms")
        show_image(annotated, title)

    return annotated, detections, meta


print("✅ Single-image pipeline defined.")

## Cell 10 — Batch Processing Pipeline

In [ ]:
def process_batch(input_folder, main_model, person_model,
                   extensions=(".jpg", ".jpeg", ".png", ".bmp", ".webp"),
                   show_first_n=3):
    """
    Process all images in a folder.
    Returns a DataFrame with detection metadata.
    """
    input_folder = Path(input_folder)
    all_images   = sorted([
        p for p in input_folder.iterdir()
        if p.suffix.lower() in extensions
    ])

    if not all_images:
        print(f"⚠️  No images found in: {input_folder}")
        return pd.DataFrame()

    print(f"📂 Found {len(all_images)} images in '{input_folder}'")
    print(f"   Saving annotated results to: {OUTPUT_DIR}/annotated/\n")

    all_meta = []
    all_detections = []   # flat list for full detail export

    for idx, img_path in enumerate(tqdm(all_images, desc="Detecting")):
        show_inline = (idx < show_first_n)   # only show first N images inline
        try:
            _, dets, meta = process_single_image(
                img_path, main_model, person_model,
                save_annotated=True, show=show_inline
            )
            all_meta.append(meta)
            for d in dets:
                all_detections.append({"filename": img_path.name, **d})
        except Exception as e:
            print(f"   ❌ Error on {img_path.name}: {e}")

    # Save raw detections JSON
    json_path = os.path.join(OUTPUT_DIR, "reports", "raw_detections.json")
    with open(json_path, "w") as f:
        # Convert tuples to lists for JSON serialisation
        serialisable = [
            {**d, "bbox": list(d["bbox"])} for d in all_detections
        ]
        json.dump(serialisable, f, indent=2)

    df = pd.DataFrame(all_meta)
    csv_path = os.path.join(OUTPUT_DIR, "reports", "summary.csv")
    df.to_csv(csv_path, index=False)

    print(f"\n✅ Batch complete.")
    print(f"   Summary CSV  : {csv_path}")
    print(f"   Raw JSON     : {json_path}")

    return df


print("✅ Batch pipeline defined.")

## Cell 11 — Run: Single Image Test
> **Usage:** Set `TEST_IMAGE` to any traffic image path and run.

In [ ]:
# ── Change this path to your test image ──
TEST_IMAGE = "./sample_traffic.jpg"   # e.g., "/content/traffic_cam_001.jpg"

if os.path.exists(TEST_IMAGE):
    annotated_img, detections, metadata = process_single_image(
        TEST_IMAGE, main_model, person_model,
        save_annotated=True, show=True
    )

    print("\n📋 Detection Summary:")
    print(f"   File       : {metadata['filename']}")
    print(f"   Size       : {metadata['image_size']}")
    print(f"   Inference  : {metadata['inference_ms']} ms")
    print(f"   Detections : {metadata['total_detected']}")
    print("\n📦 Per-object details:")
    for i, d in enumerate(detections, 1):
        x1,y1,x2,y2 = d['bbox']
        print(f"   [{i:02d}] {d['class_name']:12s}  conf={d['confidence']:.3f}  "
              f"box=({x1},{y1},{x2},{y2})  cat={d['category']}")
else:
    print(f"⚠️  Image not found: {TEST_IMAGE}")
    print("     Please update TEST_IMAGE to point to a valid traffic image.")

## Cell 12 — Run: Batch Processing

In [ ]:
# ── Change INPUT_PATH to your folder of traffic images ──
if os.path.isdir(INPUT_PATH):
    summary_df = process_batch(
        INPUT_PATH, main_model, person_model,
        show_first_n=3    # show first 3 inline; rest saved only
    )
    if not summary_df.empty:
        print("\n📊 Summary Table (first 10 rows):")
        display(summary_df.head(10))
else:
    print(f"⚠️  Input folder not found: {INPUT_PATH}")
    print("     Please set INPUT_PATH to the folder containing your traffic images.")
    summary_df = pd.DataFrame()

## Cell 13 — Vehicle Category Classification Report

In [ ]:
def plot_class_distribution(summary_df):
    """Bar chart of detected object counts across all images."""
    if summary_df.empty:
        print("No data to plot. Run batch processing first.")
        return

    count_cols = [c for c in summary_df.columns if c.startswith("count_")]
    totals = summary_df[count_cols].sum().sort_values(ascending=False)
    totals.index = [c.replace("count_", "") for c in totals.index]

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    # Bar chart
    colors = ["#FF6B35" if c in ("person", "rider") else "#3D85C8"
              for c in totals.index]
    axes[0].bar(totals.index, totals.values, color=colors, edgecolor='white', linewidth=0.8)
    axes[0].set_title("Total Detections by Class", fontsize=13, fontweight='bold')
    axes[0].set_xlabel("Class")
    axes[0].set_ylabel("Count")
    axes[0].tick_params(axis='x', rotation=30)
    for i, (k, v) in enumerate(totals.items()):
        axes[0].text(i, v + 0.5, str(int(v)), ha='center', fontsize=10)

    # Pie chart — vehicles vs road users
    vehicle_keys  = ["car", "bus", "truck", "motorcycle", "bicycle"]
    road_user_keys = ["person", "rider"]
    veh_total = sum(totals.get(k, 0) for k in vehicle_keys)
    usr_total = sum(totals.get(k, 0) for k in road_user_keys)

    pie_vals   = [veh_total, usr_total]
    pie_labels = [f"Vehicles\n({int(veh_total)})", f"Road Users\n({int(usr_total)})"]
    axes[1].pie(pie_vals, labels=pie_labels, autopct='%1.1f%%',
                colors=["#3D85C8", "#FF6B35"], startangle=90,
                textprops={'fontsize': 11})
    axes[1].set_title("Vehicles vs Road Users", fontsize=13, fontweight='bold')

    plt.suptitle("Vehicle & Road User Detection — Class Distribution",
                 fontsize=15, fontweight='bold', y=1.02)
    plt.tight_layout()
    plot_path = os.path.join(OUTPUT_DIR, "reports", "class_distribution.png")
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"   Chart saved: {plot_path}")


plot_class_distribution(summary_df)

## Cell 14 — Inference Speed Analysis

In [ ]:
def plot_inference_speed(summary_df):
    """Box + strip plot of per-image inference time."""
    if summary_df.empty or "inference_ms" not in summary_df.columns:
        print("No timing data. Run batch processing first.")
        return

    fig, ax = plt.subplots(figsize=(10, 4))
    ms_vals = summary_df["inference_ms"].dropna()

    ax.boxplot(ms_vals, vert=False, patch_artist=True,
               boxprops=dict(facecolor='#3D85C8', alpha=0.6),
               medianprops=dict(color='orange', linewidth=2))
    ax.scatter(ms_vals, [1]*len(ms_vals), alpha=0.5, color='#FF6B35', zorder=3, s=30)

    ax.set_xlabel("Inference Time (ms)", fontsize=12)
    ax.set_title(f"Per-Image Inference Speed  |  median={ms_vals.median():.0f} ms  "
                 f" avg={ms_vals.mean():.0f} ms",
                 fontsize=13, fontweight='bold')
    ax.set_yticks([])
    ax.axvline(1000/30, color='green', linestyle='--', linewidth=1.2, label='30 FPS line')
    ax.legend()
    plt.tight_layout()
    plt.show()

    print(f"   Min: {ms_vals.min():.0f} ms    Max: {ms_vals.max():.0f} ms    "
          f"Mean: {ms_vals.mean():.0f} ms    Median: {ms_vals.median():.0f} ms")


plot_inference_speed(summary_df)

## Cell 15 — Crop & Save Individual Detections

In [ ]:
def save_detection_crops(image_path, detections, output_dir, padding=10):
    """
    Crop and save each detected object as an individual image.
    Useful for downstream tasks (helmet check, plate OCR, etc.).
    """
    img_bgr, _ = load_image(image_path)
    h, w = img_bgr.shape[:2]
    stem = Path(image_path).stem

    saved = []
    for i, det in enumerate(detections):
        x1, y1, x2, y2 = det["bbox"]
        # Add padding (clamp to image bounds)
        x1c = max(0, x1 - padding)
        y1c = max(0, y1 - padding)
        x2c = min(w, x2 + padding)
        y2c = min(h, y2 + padding)

        crop = img_bgr[y1c:y2c, x1c:x2c]
        class_dir = os.path.join(output_dir, "crops", det["class_name"])
        os.makedirs(class_dir, exist_ok=True)

        crop_path = os.path.join(class_dir, f"{stem}_det{i:03d}.jpg")
        cv2.imwrite(crop_path, crop)
        saved.append(crop_path)

    return saved


# Demo on the test image
if os.path.exists(TEST_IMAGE) and 'detections' in dir():
    crop_paths = save_detection_crops(TEST_IMAGE, detections, OUTPUT_DIR)
    print(f"✅ Saved {len(crop_paths)} crops to: {OUTPUT_DIR}/crops/")

    # Display a grid of crops
    n = min(len(crop_paths), 12)
    if n > 0:
        cols = 4
        rows = (n + cols - 1) // cols
        fig, axes = plt.subplots(rows, cols, figsize=(cols*3, rows*3))
        axes = np.array(axes).flatten()
        for idx, cp in enumerate(crop_paths[:n]):
            img = cv2.cvtColor(cv2.imread(cp), cv2.COLOR_BGR2RGB)
            axes[idx].imshow(img)
            axes[idx].set_title(Path(cp).parent.name, fontsize=9)
            axes[idx].axis('off')
        for ax in axes[n:]:
            ax.axis('off')
        plt.suptitle("Detection Crops", fontsize=13, fontweight='bold')
        plt.tight_layout()
        plt.show()
else:
    print("⚠️  Skipping crop demo — run Cell 11 first with a valid test image.")

## Cell 16 — Model Comparison (n / s / m variants)

In [ ]:
def compare_model_variants(image_path, models_dir, variants=("n", "s", "m")):
    """
    Run each YOLOv8 variant on the same image and compare:
    - Number of detections
    - Average confidence
    - Inference time
    """
    if not os.path.exists(image_path):
        print(f"⚠️  Image not found: {image_path}")
        return

    img_bgr, _ = load_image(image_path)
    img_pre = preprocess_image(img_bgr)

    records = []
    annotated_imgs = []

    for var in variants:
        model_path = os.path.join(models_dir, f"yolov8{var}.pt")
        try:
            model_v = load_model(model_path, device)
        except Exception as e:
            print(f"  Skipping {var}: {e}")
            continue

        t0 = time.time()
        dets = detect_traffic_objects(img_pre, model_v, model_v,
                                       conf=CONF_THRESHOLD, iou=IOU_THRESHOLD)
        elapsed_ms = (time.time() - t0) * 1000

        avg_conf = np.mean([d["confidence"] for d in dets]) if dets else 0.0
        records.append({
            "Variant": f"yolov8{var}",
            "Detections": len(dets),
            "Avg Confidence": round(avg_conf, 3),
            "Inference (ms)": round(elapsed_ms, 1),
        })
        annotated_imgs.append((f"yolov8{var}", draw_detections(img_bgr, dets)))

    # Print comparison table
    comp_df = pd.DataFrame(records)
    print("\n📊 Model Comparison:")
    display(comp_df)

    # Side-by-side display
    n = len(annotated_imgs)
    fig, axes = plt.subplots(1, n, figsize=(7*n, 6))
    if n == 1:
        axes = [axes]
    for ax, (name, ann) in zip(axes, annotated_imgs):
        ax.imshow(cv2.cvtColor(ann, cv2.COLOR_BGR2RGB))
        ax.set_title(name, fontsize=12, fontweight='bold')
        ax.axis('off')
    plt.suptitle("YOLOv8 Variant Comparison", fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

    return comp_df


# Uncomment to run the comparison:
# compare_model_variants(TEST_IMAGE, MODELS_DIR, variants=("n", "s", "m"))
print("✅ Model comparison function defined. Uncomment the last line to run.")

## Cell 17 — Performance Metrics (if ground truth labels available)

In [ ]:
def compute_metrics_from_yolo(model, dataset_yaml_path,
                               conf=0.25, iou=0.50, imgsz=640):
    """
    If you have a labelled dataset in YOLO format, this computes:
    mAP@50, mAP@50-95, Precision, Recall, F1 per class.

    dataset_yaml_path : path to dataset.yaml (YOLO dataset descriptor)
    """
    if not os.path.exists(dataset_yaml_path):
        print(f"⚠️  dataset.yaml not found: {dataset_yaml_path}")
        print("   Skipping metrics — provide a labelled YOLO-format dataset.")
        return None

    print("⏳ Running validation...")
    metrics = model.val(
        data=dataset_yaml_path,
        conf=conf,
        iou=iou,
        imgsz=imgsz,
        verbose=True,
        device=device,
    )

    print("\n📐 Evaluation Metrics:")
    print(f"   mAP@50      : {metrics.box.map50:.4f}")
    print(f"   mAP@50-95   : {metrics.box.map:.4f}")
    print(f"   Precision   : {metrics.box.mp:.4f}")
    print(f"   Recall      : {metrics.box.mr:.4f}")

    f1 = 2 * metrics.box.mp * metrics.box.mr / \
         (metrics.box.mp + metrics.box.mr + 1e-8)
    print(f"   F1-score    : {f1:.4f}")

    return metrics


# Example usage (comment out if no labelled dataset):
# DATASET_YAML = "./datasets/traffic/dataset.yaml"
# metrics = compute_metrics_from_yolo(main_model, DATASET_YAML)

print("✅ Metrics function defined.")
print("   To evaluate, set DATASET_YAML to your dataset.yaml path and call compute_metrics_from_yolo().")

## Cell 18 — Video Processing (Optional)

In [ ]:
def process_video(video_path, main_model, person_model,
                   output_video_path=None, max_frames=None):
    """
    Run detection on every frame of a video and write annotated output.
    """
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        print(f"❌ Cannot open video: {video_path}")
        return

    fps    = int(cap.get(cv2.CAP_PROP_FPS))
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if max_frames:
        total = min(total, max_frames)

    print(f"🎬 Video: {video_path}")
    print(f"   {width}x{height} @ {fps} fps  |  {total} frames to process")

    out_path = output_video_path or os.path.join(
        OUTPUT_DIR, "annotated", Path(video_path).stem + "_detected.mp4")
    writer = cv2.VideoWriter(
        out_path,
        cv2.VideoWriter_fourcc(*'mp4v'),
        fps, (width, height)
    )

    frame_idx = 0
    pbar = tqdm(total=total, desc="Frames")

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret or (max_frames and frame_idx >= max_frames):
            break

        processed = preprocess_image(frame)
        dets = detect_traffic_objects(processed, main_model, person_model)
        dets = associate_riders(dets)
        annotated = draw_detections(frame, dets)

        # Overlay frame number and FPS
        cv2.putText(annotated, f"Frame {frame_idx}", (width-160, 24),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (200,200,200), 1)

        writer.write(annotated)
        frame_idx += 1
        pbar.update(1)

    cap.release()
    writer.release()
    pbar.close()
    print(f"\n✅ Video saved: {out_path}")


# Example usage:
# process_video("./traffic_video.mp4", main_model, person_model, max_frames=300)
print("✅ Video pipeline defined. Uncomment the last line to process a video.")

## Cell 19 — Final Summary & Next Steps

In [ ]:
print("=" * 60)
print("  STEP 2 — Vehicle & Road User Detection — COMPLETE")
print("=" * 60)

print("""
✅ What this notebook covers:
   • YOLOv8 pretrained model loading (main + person models)
   • CLAHE image preprocessing for low-light / shadow robustness
   • Dual-model detection pipeline (vehicles + persons)
   • Rider ↔ vehicle spatial association
   • Triple-riding early flag
   • Annotated image output (bounding boxes + labels + summary)
   • Crop extraction per detected object
   • Batch processing over a folder of images
   • Class distribution + inference speed charts
   • Model variant comparison (n / s / m)
   • Video processing pipeline
   • mAP / Precision / Recall / F1 evaluation (if labels available)

📁 Outputs saved to:
   {}/
   ├── annotated/     ← annotated images
   ├── crops/         ← per-class object crops
   └── reports/
       ├── summary.csv          ← per-image stats
       ├── raw_detections.json  ← all bounding boxes + metadata
       └── class_distribution.png

🔜 Next Steps (Step 3 — Violation Detection):
   • Use cropped 'rider' images  → helmet compliance check
   • Use cropped 'car/truck'     → seatbelt check
   • Use rider_count per vehicle → triple-riding rule
   • Add lane / stop-line ROI    → wrong-side / red-light violations
   • Feed 'car/motorcycle' crops → LPR / OCR for plate recognition
""".format(OUTPUT_DIR))

print("=" * 60)